# Phase 11 — Robust Error Handling & Circuit Breaker

> Run AFTER Phase 11 (restart kernel first). No Gemini calls needed for cells 1-4 (real yfinance network calls only); cell 5 uses Gemini for the full end-to-end proof.

**Goal:** point a secondary adapter at a dead URL, fire 5 queries, watch the circuit open on the 3rd and yfinance take over — the user never sees an error.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv('../.env')
assert os.getenv('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY in .env'
print('env OK')

env OK


## 1. Circuit breaker mechanics — closed → open → half-open → closed

In [2]:
import time
from app.errors.circuit_breaker import CircuitBreaker, CircuitOpenError
cb = CircuitBreaker('demo', threshold=3, cooldown=0.3)
def always_fails(): raise RuntimeError('adapter down')
for i in range(1, 5):
    try:
        cb.call(always_fails)
    except CircuitOpenError:
        print(f'attempt {i}: OPEN — failed fast, no call made. state={cb.state.value}')
    except RuntimeError:
        print(f'attempt {i}: real call attempted and failed. state={cb.state.value}')
time.sleep(0.35)
print('after cooldown:', cb.state.value)
print('recovery call  :', cb.call(lambda: 'recovered'), '-> state:', cb.state.value)

attempt 1: real call attempted and failed. state=closed
attempt 2: real call attempted and failed. state=closed
{"adapter": "demo", "failures": 3, "event": "circuit_opened", "level": "warning", "timestamp": "2026-07-13T10:12:45.786115Z"}


attempt 3: real call attempted and failed. state=open
{"adapter": "demo", "failures": 3, "event": "circuit_fail_fast", "level": "warning", "timestamp": "2026-07-13T10:12:45.786531Z"}


attempt 4: OPEN — failed fast, no call made. state=open


after cooldown: half_open
{"adapter": "demo", "event": "circuit_closed", "level": "info", "timestamp": "2026-07-13T10:12:46.137620Z"}


recovery call  : recovered -> state: closed


## 2. Fallback — primary → secondary on any failure

In [3]:
from app.errors.fallback import Fallback
result = Fallback(lambda: (_ for _ in ()).throw(ValueError('primary down')),
                   lambda: 'secondary result').run()
print(result)

{"name": "fallback", "error": "primary down", "event": "fallback_engaged", "level": "warning", "timestamp": "2026-07-13T10:12:46.149264Z"}


secondary result


## 3. THE ACCEPTANCE DEMO — point Alpha Vantage at a dead URL, run 5 real market queries

In [4]:
import app.integrations.alpha_vantage_adapter as av_mod
av_mod.BASE_URL = 'https://this-host-does-not-exist-12345.invalid/query'

from app.integrations.chain import MarketDataChain
from app.integrations.alpha_vantage_adapter import AlphaVantageAdapter
from app.integrations.yfinance_adapter import YFinanceAdapter
from app.errors.circuit_breaker import circuit_breakers

av, yf = AlphaVantageAdapter(), YFinanceAdapter()
circuit_breakers._breakers.pop(av.name, None)
chain = MarketDataChain([av, yf])

for i in range(1, 6):
    q = chain.get_quote('NVDA')   # the user-facing call — must NEVER raise
    breaker = circuit_breakers.get(av.name)
    print(f'call {i}: price=${q["price"]:<8} served_by={q["source"]:10} '
          f'AlphaVantage_circuit={breaker.state.value}')

{"method": "get_quote", "adapter": "alpha_vantage", "error": "HTTPSConnectionPool(host='this-host-does-not-exist-12345.invalid', port=443): Max retries exceeded w", "event": "chain_adapter_failed", "level": "warning", "timestamp": "2026-07-13T10:12:46.481121Z"}


{"method": "get_quote", "source": "yfinance", "event": "chain_served", "level": "info", "timestamp": "2026-07-13T10:12:47.550937Z"}


call 1: price=$210.96   served_by=yfinance   AlphaVantage_circuit=closed
{"method": "get_quote", "adapter": "alpha_vantage", "error": "HTTPSConnectionPool(host='this-host-does-not-exist-12345.invalid', port=443): Max retries exceeded w", "event": "chain_adapter_failed", "level": "warning", "timestamp": "2026-07-13T10:12:47.552694Z"}


{"method": "get_quote", "source": "yfinance", "event": "chain_served", "level": "info", "timestamp": "2026-07-13T10:12:47.781641Z"}


call 2: price=$210.96   served_by=yfinance   AlphaVantage_circuit=closed
{"adapter": "alpha_vantage", "failures": 3, "event": "circuit_opened", "level": "warning", "timestamp": "2026-07-13T10:12:47.783091Z"}


...

{"method": "get_quote", "source": "yfinance", "event": "chain_served", "level": "info", "timestamp": "2026-07-13T10:12:48.187439Z"}


call 4: price=$210.96   served_by=yfinance   AlphaVantage_circuit=open
{"adapter": "alpha_vantage", "failures": 3, "event": "circuit_fail_fast", "level": "warning", "timestamp": "2026-07-13T10:12:48.187817Z"}


{"method": "get_quote", "adapter": "alpha_vantage", "event": "chain_circuit_skip", "level": "info", "timestamp": "2026-07-13T10:12:48.188059Z"}


{"method": "get_quote", "source": "yfinance", "event": "chain_served", "level": "info", "timestamp": "2026-07-13T10:12:48.383600Z"}


call 5: price=$210.96   served_by=yfinance   AlphaVantage_circuit=open


## 4. Retriever fallback — ChromaDB down → keyword search over data/knowledge_base/

In [5]:
from app.knowledge.retriever import Retriever
from app.knowledge.vector_store import VectorStore

class DeadStore(VectorStore):
    def __init__(self): pass
    def query(self, *a, **k): raise ConnectionError('Chroma is down (simulated)')

r = Retriever(store=DeadStore())
results = r.query('data center demand', filters={'ticker': 'NVDA'}, k=3)
print(f'{len(results)} result(s) served via keyword fallback (Chroma never touched successfully)')
if results:
    print('first hit:', results[0][0][:100], '...')

{"name": "retriever", "error": "Chroma is down (simulated)", "event": "fallback_engaged", "level": "warning", "timestamp": "2026-07-13T10:12:49.327762Z"}


{"collection": "sec_filings", "event": "retrieval_fallback_engaged", "level": "warning", "timestamp": "2026-07-13T10:12:49.328264Z"}


{"query": "data center demand", "hits": 3, "event": "keyword_fallback_search", "level": "info", "timestamp": "2026-07-13T10:12:49.370047Z"}


{"collection": "sec_filings", "hits": 3, "filters": {"ticker": "NVDA"}, "event": "retrieval", "level": "info", "timestamp": "2026-07-13T10:12:49.370552Z"}


3 result(s) served via keyword fallback (Chroma never touched successfully)
first hit: Data Center
The NVIDIA Data Center platform is focused on accelerating compute-intensive workloads,  ...


## 5. End-to-end: an agent that always crashes still yields a safe answer, no stack trace

In [6]:
from langchain_core.messages import HumanMessage
from app.agents.portfolio import PortfolioAgent
from app.graph.builder import GraphBuilder
from app.graph.router import KeywordRoutingStrategy

def crash(self, state): raise RuntimeError('simulated LLM outage')
orig = PortfolioAgent._invoke
PortfolioAgent._invoke = crash
try:
    graph = (GraphBuilder().with_supervisor(strategy=KeywordRoutingStrategy())
             .with_portfolio_agent().with_guardrails().build())
    result = graph.invoke({'messages':[HumanMessage(content='What do I own?')],
                           'client_id':'CLT-001','session_id':'nb11'})
    print('USER SEES:', result['messages'][-1].content)
finally:
    PortfolioAgent._invoke = orig

{"guard": "pii", "action": "pass", "passed": true, "reason": "", "event": "guardrail_check", "client_id": "CLT-001", "agent": "input_guard", "session_id": "nb11", "level": "info", "timestamp": "2026-07-13T10:12:50.423173Z"}


{"guard": "prompt_injection", "action": "pass", "passed": true, "reason": "", "event": "guardrail_check", "client_id": "CLT-001", "agent": "input_guard", "session_id": "nb11", "level": "info", "timestamp": "2026-07-13T10:12:50.423625Z"}


{"guard": "scope", "action": "pass", "passed": true, "reason": "", "event": "guardrail_check", "client_id": "CLT-001", "agent": "input_guard", "session_id": "nb11", "level": "info", "timestamp": "2026-07-13T10:12:50.423957Z"}


{"next_agent": "portfolio", "hop": 1, "event": "supervisor_dispatch", "client_id": "CLT-001", "agent": "supervisor", "session_id": "nb11", "level": "info", "timestamp": "2026-07-13T10:12:50.424711Z"}


{"query": "What do I own?", "event": "agent_start", "client_id": "CLT-001", "agent": "portfolio", "session_id": "nb11", "level": "info", "timestamp": "2026-07-13T10:12:50.425547Z"}


{"tool": "portfolio", "error_type": "RuntimeError", "client_id": "CLT-001", "session_id": "nb11", "error": "simulated LLM outage", "event": "tool_error", "agent": "portfolio", "level": "error", "timestamp": "2026-07-13T10:12:50.425888Z"}


{"event": "safe_exit", "level": "info", "timestamp": "2026-07-13T10:12:50.426424Z"}


USER SEES: I couldn't complete that because the portfolio step ran into a problem. Please try rephrasing your question.


## ✅ Acceptance check

In [7]:
assert circuit_breakers.get(av.name).state.value == 'open'
assert all(chain.get_quote('NVDA')['price'] > 0 for _ in range(2))  # still never errors
print('Phase 11 acceptance: PASS — circuit opened on attempt 3, fallback engaged, no user-visible errors')

{"adapter": "alpha_vantage", "failures": 3, "event": "circuit_fail_fast", "level": "warning", "timestamp": "2026-07-13T10:12:50.443622Z"}


{"method": "get_quote", "adapter": "alpha_vantage", "event": "chain_circuit_skip", "level": "info", "timestamp": "2026-07-13T10:12:50.444051Z"}


{"method": "get_quote", "source": "yfinance", "event": "chain_served", "level": "info", "timestamp": "2026-07-13T10:12:50.646859Z"}


{"adapter": "alpha_vantage", "failures": 3, "event": "circuit_fail_fast", "level": "warning", "timestamp": "2026-07-13T10:12:50.647458Z"}


{"method": "get_quote", "adapter": "alpha_vantage", "event": "chain_circuit_skip", "level": "info", "timestamp": "2026-07-13T10:12:50.647688Z"}


{"method": "get_quote", "source": "yfinance", "event": "chain_served", "level": "info", "timestamp": "2026-07-13T10:12:50.875954Z"}


Phase 11 acceptance: PASS — circuit opened on attempt 3, fallback engaged, no user-visible errors
